# Module 1: Optimization Problems & Karush-Kuhn-Tucker (KKT) Conditions


In this module we investigate **optimization basics** (linear and nonlinear programs), notions of **convex functions**, **convex sets**, and **Karush-Kuhn-Tucker optimality conditions**.

We will investigate these concepts through the following examples:
1)  Transportation Problem (LP)

2)  Portfolio Optimization (NLP)

3)  Convexity Analysis

#### Activate the course environment and import packages

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

using JuMP, HiGHS, Ipopt, Plots, DataFrames

# Exercise 1.1: Transportation Problem

#### Problem Description

A company operates 3 hydrogen production plants and must supply hydrogen to 4 industrial customers. Each hydrogen production plant has a limited production capacity, and each industrial customer has a minimum demand requirement. The company wants to minimize the total cost of distributing hydrogen while meeting all contracts.

##### Problem Data:
- **Supply**: 3 plants
- **Demand**: 4 customers
- **Objective**: Minimize total distribution cost

### Task 1: Implement transportation model in JuMP

#### Explicit mathematical formulation:

**Decision Variables:**
- $x_{ij}$ = amount of hydrogen distributed from plant $i$ to customer $j$

**Objective Function:**
$$\min Z = 8x_{11} + 6x_{12} + 10x_{13} + 9x_{14} + 9x_{21} + 12x_{22} + 13x_{23} + 7x_{24} + 14x_{31} + 9x_{32} + 16x_{33} + 5x_{34}$$

**Subject to:**

*Supply Constraints:*
- $x_{11} + x_{12} + x_{13} + x_{14} \leq 35$ (Plant 1 capacity)
- $x_{21} + x_{22} + x_{23} + x_{24} \leq 50$ (Plant 2 capacity)
- $x_{31} + x_{32} + x_{33} + x_{34} \leq 40$ (Plant 3 capacity)

*Demand Constraints:*
- $x_{11} + x_{21} + x_{31} \geq 45$ (Customer 1 demand)
- $x_{12} + x_{22} + x_{32} \geq 20$ (Customer 2 demand)
- $x_{13} + x_{23} + x_{33} \geq 30$ (Customer 3 demand)
- $x_{14} + x_{24} + x_{34} \geq 30$ (Customer 4 demand)

*Non-negativity:*
- $x_{ij} \geq 0$ for all $i,j$

#### Symbolic formulation

\begin{align*}
\min & \sum_{i \in I} \sum_{j \in J} c_{ij}x_{ij} \\
\text{s.t.} \quad & \sum_{i \in I} x_{ij} \geq d_j \qquad \forall j \in J \\
& \sum_{j \in J} x_{ij} \leq s_i, \qquad \forall i \in I \\
& x_{ij} \geq 0, \qquad \forall i \in I, j \in J
\end{align*}

Where:
- $x_{ij}$ = amount of hydrogen (kg) distributed from plant $i$ to customer $j$
- $c_{ij}$ = transportation cost (€/kg) for sending from plant $i$ to customer $j$
- $d_j$    = hydrogen demand (kg) for customer $j$
- $s_i$    = hydrogen supply (kg) for customer $i$
- $I$      = set of plants
- $J$      = set of customers

#### Define Problem Data

In [ ]:
# Define sets of plants and customers
plants = ["Plant_1", "Plant_2", "Plant_3"]
customers = ["Customer_1", "Customer_2", "Customer_3", "Customer_4"]

# Supply capacities (kg)
supply_capacities = Dict(
    "Plant_1" => 35,
    "Plant_2" => 50,
    "Plant_3" => 40
)

# Demand requirements (kg)
demand_quantities = Dict(
    "Customer_1" => 45,
    "Customer_2" => 20,
    "Customer_3" => 30,
    "Customer_4" => 30
)

# Arc transportation costs ($/kg)
transportation_costs = Dict(
    ("Plant_1","Customer_1") => 8, ("Plant_1","Customer_2") => 6, ("Plant_1","Customer_3") => 10, ("Plant_1","Customer_4") => 9,
    ("Plant_2","Customer_1") => 9, ("Plant_2","Customer_2") => 12, ("Plant_2","Customer_3") => 13, ("Plant_2","Customer_4") => 7,
    ("Plant_3","Customer_1") => 14, ("Plant_3","Customer_2") => 9, ("Plant_3","Customer_3") => 16, ("Plant_3","Customer_4") => 5
)

# Node positions for plotting later (plants on left, customers on right)
plant_coordinates = Dict(p => (0, 3 - (idx-1)) for (idx, p) in enumerate(plants))
customer_coordinates = Dict(c => (4, 3.5 - (idx-1)) for (idx, c) in enumerate(customers))
coordinates = merge(plant_coordinates, customer_coordinates);

#### Function that constructs the model for specified data

We choose to use the **HiGHS** solver for this problem. It is a free and open-source solver and is suitable for solving linear programs.

In [ ]:
function construct_transportation_model(plants, customers, supply_capacities, demand_quantities, transport_costs)

    # Create the model
    model = Model(HiGHS.Optimizer)

    # Define decision variables (indexed over plants and customers)
    @variable(model, x[plants, customers] >= 0)

    # Define objective function
    @objective(model, Min, sum(transport_costs[i,j] * x[i,j] for i in plants, j in customers))

    # Define supply constraints
    @constraint(model, supply_con[i in plants], sum(x[i,j] for j in customers) <= supply_capacities[i])

    # Define demand constraints
    @constraint(model, demand_con[j in customers], sum(x[i,j] for i in plants) >= demand_quantities[j])

    # Return model object
    return model
end

#### Construct the model using the function and print the model

In [ ]:
tp_model = construct_transportation_model(plants, customers, supply_capacities, demand_quantities, transportation_costs)

# Print model to verify correctness
print(tp_model)

#### Solve the model and display results

In [ ]:
# Solve the model 
optimize!(tp_model)

Observe solution status, objective value, and decision variable values.

In [ ]:
println("=== Objective value ===")
println("Total Transportation Cost: ", round(objective_value(tp_model), digits=2), " €")

println("\n=== Optimal Transportation Plan ===")
x = tp_model[:x]
for i in plants, j in customers
    if value(x[i,j]) > 1e-6
        println("Transport ", round(value(x[i,j]), digits=2), " from ", i, " to ", j)
    end
end

println("\n=== Dual Variables ===")
# Function for extracting supply constraint duals.
# The sign is flipped so it reads as the value ($/kg) relaxing capacity by one unit.
function supply_duals(model)
    return Dict(i => 
    dual(model[:supply_con][i]) 
    for i in plants)
end

# Function for extracting demand constraint duals
function demand_duals(model)
    return Dict(j => dual(model[:demand_con][j]) for j in customers)
end

println("Supply Constraints (Shadow Prices for Plants, €/kg")
plant_duals = supply_duals(tp_model)
for i in plants
    println(i, ": ", plant_duals[i], " €/kg")
end

println("\nDemand Constraints (Shadow Prices for Customers, €/kg")
customer_duals = demand_duals(tp_model)
for j in customers
    println(j, ": ", customer_duals[j], " €/kg")
end

#### Visualize results

The function below visualizes the transportation plan, showing the flow of hydrogen from plants to customers.

In [ ]:
function plot_tp_result(model, costs, supply_duals, demand_duals;
                         coordinates=coordinates, plants=plants, customers=customers)
    x = model[:x]

    # Scale up coordinates for more breathing room between nodes (more
    # vertical than horizontal, since that's where labels need room)
    xscale, yscale = 2.6, 3.4
    coords = Dict(k => (xscale*v[1], yscale*v[2]) for (k,v) in coordinates)

    p = plot(legend=false, framestyle=:none, size=(950, 780),
             xlims=(-2.2, 12.5), ylims=(1.0, 14.1),
             ticks=nothing, margin=1Plots.mm)

    # Helper: draw a small background box behind text so labels stay legible
    # where multiple flow lines cross each other.
    function draw_label_box!(p, x0, y0, str, fontsize, color, w, h)
        box = Shape([x0-w/2, x0+w/2, x0+w/2, x0-w/2], [y0-h/2, y0-h/2, y0+h/2, y0+h/2])
        plot!(p, box, color=:white, linecolor=RGBA(0,0,0,0.15), linewidth=0.5, alpha=0.95, label=false)
        annotate!(p, x0, y0, text(str, fontsize, color))
    end

    flows = [(i,j) for i in plants, j in customers if value(x[i,j]) > 1e-6]

    # Stagger each label's initial position along its line (as a fraction t)
    # based on how many other edges leave the same plant / enter the same
    # customer, then run a generic pairwise decluttering pass below to nudge
    # apart any labels that still collide (e.g. because two unrelated edges
    # happen to cross near the same point).
    out_idx = Dict{String,Int}()
    in_idx = Dict{String,Int}()

    labels = NamedTuple[]  # (x, y, str, fontsize, color, w, h)
    for (i, j) in flows
        flow = value(x[i,j])
        xi, yi = coords[i]
        xj, yj = coords[j]
        plot!(p, [xi, xj], [yi, yj], color=:gray55, linewidth=3, alpha=0.75)

        oi = get!(out_idx, i, 0); out_idx[i] += 1
        ii = get!(in_idx, j, 0); in_idx[j] += 1

        t_flow = 0.28 + 0.07*oi
        t_cost = 0.68 + 0.07*ii

        fx, fy = xi + t_flow*(xj-xi), yi + t_flow*(yj-yi)
        cx, cy = xi + t_cost*(xj-xi), yi + t_cost*(yj-yi)

        flow_str = "$(round(flow, digits=1)) kg"
        cost_str = "$(costs[(i,j)])€/kg"
        push!(labels, (x=fx, y=fy, str=flow_str, fontsize=9, color=:navy,
                        w=0.16*length(flow_str)+0.15, h=0.42))
        push!(labels, (x=cx, y=cy, str=cost_str, fontsize=9, color=:gray30,
                        w=0.16*length(cost_str)+0.15, h=0.42))
    end

    # Generic pairwise decluttering: push apart any two label boxes that
    # overlap, regardless of which edges they came from.
    xs = [l.x for l in labels]; ys = [l.y for l in labels]
    for _ in 1:60
        moved = false
        for a in 1:length(labels), b in (a+1):length(labels)
            dx = xs[b] - xs[a]; dy = ys[b] - ys[a]
            overlap_x = (labels[a].w + labels[b].w)/2 - abs(dx)
            overlap_y = (labels[a].h + labels[b].h)/2 - abs(dy)
            if overlap_x > 0 && overlap_y > 0
                moved = true
                if overlap_x < overlap_y
                    shift = (overlap_x/2 + 0.01) * sign(dx == 0 ? 1.0 : dx)
                    xs[a] -= shift; xs[b] += shift
                else
                    shift = (overlap_y/2 + 0.01) * sign(dy == 0 ? 1.0 : dy)
                    ys[a] -= shift; ys[b] += shift
                end
            end
        end
        moved || break
    end

    for (k, l) in enumerate(labels)
        draw_label_box!(p, xs[k], ys[k], l.str, l.fontsize, l.color, l.w, l.h)
    end

    # Nodes: small markers with labels placed OUTSIDE the marker (so text
    # never has to be squeezed inside a shape).
    for i in plants
        xi, yi = coords[i]
        scatter!(p, [xi], [yi], markersize=20, color=:steelblue, markerstrokewidth=1.5, markerstrokecolor=:white)
        annotate!(p, xi - 0.4, yi, text("$i\n$(supply_duals[i])€/kg", 10, :right, :black))
    end
    for j in customers
        xj, yj = coords[j]
        scatter!(p, [xj], [yj], markersize=20, color=:darkorange, markerstrokewidth=1.5, markerstrokecolor=:white)
        annotate!(p, xj + 0.4, yj, text("$j\n$(demand_duals[j])€/kg", 10, :left, :black))
    end

    annotate!(p, xscale*2, yscale*3.8, text("Optimal Transportation Plan", 15))

    return p
end

In [ ]:
plot_tp_result(tp_model, transportation_costs, plant_duals, customer_duals)

### Task 2: Analysing the relationship between transportation costs and nodal prices

In this exercise, you will analyze the relationship between nodal prices (dual variables) and transportation unit costs in the transportation problem. Nodal prices represent the marginal value of relaxing supply or demand constraints. Supply duals represent the marginal benefit of increasing supply capacity at a plant and demand duals represent the marginal cost of meeting additional demand at a customer.

Modify transportation cost parameters and observe how nodal prices change relative to transportation costs. How do nodal prices relate to transportation unit costs? Why?

In [ ]:
# Change these values to see how the results change
cost_data_new = Dict(
    ("Plant_1","Customer_1") => 8, 
    ("Plant_1","Customer_2") => 6, 
    ("Plant_1","Customer_3") => 10, 
    ("Plant_1","Customer_4") => 9,
    ("Plant_2","Customer_1") => 9, 
    ("Plant_2","Customer_2") => 12, 
    ("Plant_2","Customer_3") => 13, 
    ("Plant_2","Customer_4") => 7,
    ("Plant_3","Customer_1") => 14, 
    ("Plant_3","Customer_2") => 9, 
    ("Plant_3","Customer_3") => 16, 
    ("Plant_3","Customer_4") => 5
)

# Construct and solve new model
new_tp_model = construct_transportation_model(plants, customers, supply_capacities, demand_quantities, cost_data_new)
set_silent(new_tp_model)
optimize!(new_tp_model)

println("=== Objective value ===")
println("Total Transportation Cost: ", round(objective_value(new_tp_model), digits=2), " €")

# Fetch optimal dual variable values
new_supply_duals = supply_duals(new_tp_model)
new_demand_duals = demand_duals(new_tp_model)

# Plot the result
plot_tp_result(new_tp_model, cost_data_new, new_supply_duals, new_demand_duals)

# Exercise 1.2 : Portfolio Optimization

#### Problem Description

In the classic **Markowitz Mean-Variance Optimization** problem, the objective is to construct an investment portfolio using 3 stocks. The goal is to minimize portfolio risk (variance) while achieving a minimum expected return.

#### Problem Data:
- **Assets**: 3 stocks with different expected returns
- **Risk**: Measured by portfolio variance using variance-covariance matrix
- **Objective**: Minimize portfolio variance for given return levels

#### Mathematical Formulation:

\begin{align*}
\min & \sum_{i \in I} \sum_{j \in J} v_{ij} x_i x_j = x^T V x \\
\text{s.t.} \quad & \sum_{i \in I} x_i = 1  && \text{Budget constraint: weights sum to 100\% } \\
& \sum_{i \in I} r_i x_i \geq \textit{$\rho$} \qquad  && \text{Return constraint: achieve minimum required return } \\
& x_{i} \geq 0, \qquad \forall i \in I \qquad &&  \text{Non-negativity: no short selling}
\end{align*}

where:
- $x_i$ are decision variables representing weight of stock $i$ in the portfolio
- The following are data:
    - $V$ is the variance-covariance matrix
    - $v_{ij}$ is the covariance between stocks $i$ and $j$
    - $r_i$ is the expected return of stock $i$
    - $\rho$ is the minimum required return
    - $I$ is the set of stocks

Notice that this is a **quadratic programming (QP)** problem due to the quadratic objective function. We will use **Ipopt** (Interior Point OPTimizer), which is suitable for nonlinear programming problems.

#### Define Problem Data

In [ ]:
# Data from the table
returns = [10.0, 20.0, 30.0]  # Expected returns for 3 assets
cov_matrix = [
    4.2  -1.9   2.0;
   -1.9   6.7  -5.0;
    2.0  -5.0   7.0
];

### Task 1: Implement the model

Function constructs the portfolio optimization model for a specified minimum return level, expected returns and covariance matrix.

In [ ]:
function build_portfolio_model(; minimum_return=15.0, returns=returns, cov_matrix=cov_matrix)

    # Number of assets
    n_assets = length(returns)

    # Initialize model
    model = Model(Ipopt.Optimizer)

    # Variables (bounded between 0 and 1: no short-selling, no leverage)
    @variable(model, 0 <= x[1:n_assets] <= 1)

    # Objective: minimize variance
    @objective(model, Min, sum(cov_matrix[i,j] * x[i] * x[j] for i in 1:n_assets, j in 1:n_assets))

    # Constraint 1: full investment
    @constraint(model, budget, sum(x[i] for i in 1:n_assets) == 1)

    # Constraint 2: minimum return requirement
    @constraint(model, minimum_return_con, sum(returns[i] * x[i] for i in 1:n_assets) >= minimum_return)

    return model
end

Solve the model with given data and default minimum return

In [ ]:
portfolio_model = build_portfolio_model()
# Solve the model
set_silent(portfolio_model)
optimize!(portfolio_model)

In [ ]:
# Fetch optimal results
weights = value.(portfolio_model[:x])
variance = objective_value(portfolio_model)
expected_return = sum(returns[i] * weights[i] for i in eachindex(returns))

println("\n=== Optimal portfolio ===")
println("Expected return:  ", round(expected_return, digits=2), " %")
println("Risk (variance):  ", round(variance, digits=4))

println("\n=== Portfolio weights ===")
portfolio_table = DataFrame(
    Asset = ["Stock $i" for i in eachindex(returns)],
    ExpectedReturn = returns,
    Weight = round.(weights, digits=4),
    ReturnContribution = round.(returns .* weights, digits=2)
)
show(portfolio_table, allrows=true, allcols=true, summary=false, eltypes=false, show_row_number=false)



### Task 2: Analysis and Extensions

1. **Risk Tolerance Analysis**: How does the optimal portfolio change as we increase the minimum expected return? What happens to the risk of the portfolio?

2. **New Asset**: Add a fourth stock with 25% expected return and moderate correlations with existing stocks and Covariance = [1.5]
                   [-1.2]
                   [2.0]
                   [6.0]
    How does the solution compare to the original 3-stock portfolio with a minimum return of 20%?

##### **(a) Risk Tolerance Analysis**

We will investigate risk tolerance by generating the **efficient frontier** and observing how the portfolio composition and the expected return changes as we increase the minimum expected return.

In [ ]:
# Generate Efficient Frontier with custom target returns
minimum_returns = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 29, 29.5, 29.7, 29.9]

frontier_results = DataFrame(
    MinimumReturn = Float64[], ExpectedReturn = Float64[], Variance = Float64[],
    x1 = Float64[], x2 = Float64[], x3 = Float64[]
)

# Iterate over target returns and solve
for min_return in minimum_returns
    # Construct model and solve it
    model = build_portfolio_model(minimum_return=min_return)
    set_silent(model)
    optimize!(model)

    # Check if the solution is optimal
    if termination_status(model) in (LOCALLY_SOLVED, OPTIMAL)

        # Fetch optimal portfolio weights
        weights = value.(model[:x])

        # Fetch objective value
        variance = objective_value(model)

        # Calculate expected return
        expected_return = sum(returns[i] * weights[i] for i in 1:3)

        push!(frontier_results, (
            round(min_return, digits=2), round(expected_return, digits=2), round(variance, digits=4),
            round(weights[1], digits=2), round(weights[2], digits=2), round(weights[3], digits=2)
        ))
    end
end

println("="^80)
println("EFFICIENT FRONTIER RESULTS")
println("="^80)
show(frontier_results, allrows=true, allcols=true)
println()
println("="^80)

Below we visualise the efficient frontier showing the relationship between risk (variance) and return for different portfolio compositions.

In [ ]:
# Plot returns against variance (efficient frontier)
p = plot(frontier_results.ExpectedReturn, frontier_results.Variance,
         seriestype=:line, marker=:circle, linewidth=2, markersize=4,
         label="Efficient Frontier", title="Efficient Frontier",
         xlabel="Expected Return (%)", ylabel="Risk (Variance)",
         titlefontsize=14, guidefontsize=10, grid=true, gridalpha=0.3, size=(800,400), xlim=(20,35), ylim=(0,8),
         left_margin=5Plots.mm, bottom_margin=8Plots.mm)
display(p)

# Print some portfolio examples
println("="^80)
println("PORTFOLIO EXAMPLES")
println("="^80)
n = nrow(frontier_results)
example_idx = unique([1, n÷4 + 1, n÷2 + 1, 3n÷4 + 1, n])
show(frontier_results[example_idx, :], allrows=true, allcols=true)
println()
println("="^80)

##### **(b) Adding a new asset**

Add a fourth stock with 25% expected return and moderate correlations with existing stocks. Use covariance values $v_{14}=v_{41}=1.5$, $v_{24} = v_{42}=−1.2$, $v_{34} = v_{43}=2$, $v_{44}=6$. (Change the placeholder 0.0 values in the code to the correct ones.)

How does the solution of the 4-stock portfolio compare to the original 3-stock portfolio when the minimum return requirement is 20%?

In [ ]:
# Fill in the 0.0 values with appropriate values to add a fourth asset
returns_new = [10.0, 20.0, 30.0, 0.0]

cov_matrix_new = [
    4.2  -1.9   2.0   0.0;
   -1.9   6.7  -5.0   0.0;
    2.0  -5.0   7.0   0.0;
    0.0   0.0   0.0   0.0
]

# Set minimum return to 20%
min_return = 20.0

# Construct and solve new model with minimum return of 20%
new_model = build_portfolio_model(minimum_return=min_return, returns=returns_new, cov_matrix=cov_matrix_new)
set_silent(new_model)
optimize!(new_model)

# Fetch optimal results
weights = value.(new_model[:x])
variance = objective_value(new_model)
actual_return = sum(returns_new[i] * weights[i] for i in 1:4)

# Print results
println("
=== Single Run Results ===")
println("Minimum Return: ", min_return)
println("Actual Return: ", round(actual_return, digits=4))
println("Risk (Variance): ", round(variance, digits=4))
for i in 1:4
    println("Weight Asset $i: ", round(weights[i], digits=4))
end

Compare the 4-stock portfolio solution to the original 3-stock portfolio with a minimum return of 20%. The cell below runs the 3-stock portfolio optimization problem and displays the results.

In [ ]:
# Set minimum return to 20% for original 3-asset model
min_return = 20.0

# Solve the original problem
original_model = build_portfolio_model(minimum_return=min_return)
set_silent(original_model)
optimize!(original_model)

# Fetch optimal results
weights = value.(original_model[:x])
variance = objective_value(original_model)
actual_return = sum(returns[i] * weights[i] for i in 1:3)

# Print results
println("
=== Single Run Results ===")
println("Minimum Return: ", min_return)
println("Actual Return: ", round(actual_return, digits=4))
println("Risk (Variance): ", round(variance, digits=4))
for i in 1:3
    println("Weight Asset $i: ", round(weights[i], digits=4))
end

# Exercise 1.3 : Analyzing Convexity of Functions

Explore the convexity of the following five functions by plotting them in 3D and as level curves:
1. $f(x) = x^{2}$
2. $f(x) = -x^{2}$
3. $f(x) = 3x + 7 + \sin(1)$
4. $f(x,y) = xy$
5. $f(x,y) = x^{2} + y^{2}$
6. Rosenbrock function

Discuss how these visualizations reveal convexity properties.

In [ ]:
# Define grid for plotting
x = range(-5, 5, length=100)
y = range(-5, 5, length=100)

# Define the five functions
f1(x,y) = x^2
f2(x,y) = -x^2
f3(x,y) = 3x + 7 + sin(1)
f4(x,y) = x*y
f5(x,y) = x^2 + y^2

funcs = [
    (f1, "f(x, y) = x²", :viridis),
    (f2, "f(x, y) = -x²", :plasma),
    (f3, "f(x, y) = 3x + 7 + sin(1)", :magma),
    (f4, "f(x, y) = x y", :inferno),
    (f5, "f(x, y) = x² + y²", :cividis),
]

subplots = []
for (f, title_str, cmap) in funcs
    Z = [f(xi, yi) for yi in y, xi in x]

    p_surf = surface(x, y, Z, c=cmap, alpha=0.8, legend=false,
                      title=title_str, xlabel="X", ylabel="Y", zlabel="Z")
    p_cont = contour(x, y, Z, c=cmap, levels=20, colorbar=false,
                      title="Level Curves: " * title_str, xlabel="X", ylabel="Y")
    # Invisible series that only provides a solid (continuous) colour bar
    scatter!(p_cont, [NaN], [NaN], zcolor=[NaN], clims=extrema(Z), c=cmap, label=false, colorbar=true)
    push!(subplots, p_surf, p_cont)
end

plot(subplots..., layout=(5,2), size=(1200, 2000))